In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import netCDF4 as nc
import os
import gams.transfer as gt
import math
os.getcwd()

'C:\\Users\\vicke\\Desktop\\BLOEM-China\\function'

# global

In [2]:
c_id_mapping = pd.read_excel('../input/raw_data/c_index_mapping.xlsx')

In [3]:
alpha_dfa_t = np.sum([pow((1+0.1), -(p-1)) for p in range(1,11)])
alpha_dfa_t

6.759023816275149

In [4]:
# calculate dfa, dfb for each dacade
for t in range(2020, 2050, 10):
    dfa_t = alpha_dfa_t * pow( (1+0.1), -(t - 2020))
    dfb_t = pow( (1+0.1), -(t - 2020))
    print(dfa_t, dfb_t)

6.759023816275149 1.0
2.6058962754592656 0.3855432894295314
1.0046858219527297 0.14864362802414344


# function

## write and read gdx

### read_gdx(filepath)

In [5]:
def read_gdx(filepath):
    gt_file = gt.Container(filepath)
    for name, obj in gt_file:
        # print(name, obj)
        gdx_record = gt_file[name].records
    return gdx_record

### write_gdx(str_filename, dict_set, df_para, str_filedescrip)

In [6]:
def write_gdx(str_filename, dict_set, df_para, str_filedescrip):
    m = gt.Container()

    # create sets
    ls_set = []
    for set_key in dict_set:
        set_val = dict_set[set_key]
        tmp = gt.Set(m, set_key, records=set_val, description = "")
        ls_set.append(tmp)

    d = gt.Parameter(m, str_filename, ls_set, description = str_filedescrip)
    d.setRecords(df_para)
    m.write("../input/gdx/%s.gdx" % str_filename)
    return

In [7]:
# ===== write all gdx into excel ========
# cwd = '../input/gdx/'
# dir_allfilename = os.listdir(cwd)
# for filename in dir_allfilename:
#     excelname = filename.replace(".gdx", "")
#     df_tmp = read_gdx("../input/gdx/%s" % filename)
#     df_tmp.to_excel("../input/excel/%s.xlsx" % excelname)
#     print(filename)

## IMAGE_dataset

### get_time_index(ncfile, year)

In [8]:
def get_time_index(ncfile, year, str_time):
    year_float = ncfile[str_time][:].data / 365 + 1970
    list_year = list(year_float.astype(int))
    index_year = list_year.index(year)
    return index_year

### get_names(namefile_arr)

In [9]:
# get type
def get_names(namefile_arr):
    name_list = []
    for i in range(0, namefile_arr.shape[0]):
        tmp = namefile_arr[i]
        nametype = tmp[tmp!= b' ']
        str = ''
        for c in [char.decode('utf-8') for char in nametype]:
            str = str + c
        name_list.append(str)
    
    out_namefile = pd.DataFrame(name_list, columns=['name'])
    return out_namefile

### get_mapval_bycid(map_arr, c_id_mapping, map_name)

In [10]:
def get_mapval_bycid(map_arr, c_id_mapping, map_name):
    map_value = map_arr.copy()
    out_cid_df =  c_id_mapping.copy()
    out_cid_df['val'] = ''
    out_cid_df['mapname'] = ''
    for i in range(0, len(c_id_mapping)):
        col = c_id_mapping.loc[i, 'col']
        row = c_id_mapping.loc[i, 'row']
        out_cid_df.loc[i, 'val'] = map_value[row, col]
        out_cid_df.loc[i, 'mapname'] = map_name
    return out_cid_df

### get_cid_byrowcol(df, rowname, colname, c_id_mapping)

In [11]:
def get_cid_byrowcol(df, rowname, colname, c_id_mapping):
    out_df = df.copy()
    out_df['c_id'] = ''
    grid_c_mapping = c_id_mapping
    for i in df.index:
        row = df.loc[i, rowname]
        col = df.loc[i, colname]
        tmp_val = c_id_mapping.loc[(grid_c_mapping.row==row) & (grid_c_mapping.col==col), 'c_id'].values
        if len(tmp_val) == 0:
            tmp_val = c_id_mapping.loc[(grid_c_mapping.row==row) & (grid_c_mapping.col==col-1), 'c_id'].values
            if len(tmp_val) == 0:
                tmp_val = c_id_mapping.loc[(grid_c_mapping.row==row-1) & (grid_c_mapping.col==col), 'c_id'].values
                if len(tmp_val) == 0:
                     tmp_val = c_id_mapping.loc[(grid_c_mapping.row==row-1) & (grid_c_mapping.col==col-1), 'c_id'].values
        try:
            out_df.loc[i, 'c_id'] = tmp_val[0]
        except:
            print(col,row)
    return out_df

### get_rowcol_bycid(df, cid, c_id_mapping)

In [12]:
def get_rowcol_bycid(df, cid, c_id_mapping):
    out_df = df.copy()
    out_df['row'] = ""
    out_df['col'] = ""
    for i in df.index:
        c_id = df.loc[i, colstring_c_id]
        row = c_id_mapping.loc[grid_c_mapping.c_id == c_id, 'row'].values[0]
        col = c_id_mapping.loc[grid_c_mapping.c_id == c_id, 'col'].values[0]
        out_df.loc[i, 'row'] = row
        out_df.loc[i, 'col'] = col
    return out_df

## upscale dataset

### upscale_30arc_GREG_bio(bio1km)

In [13]:
def upscale_30arc_GREG_bio(bio1km):
    world_yield_zero = np.zeros((360, 720))
    shape = (80, 140)
    row_up = int((90-55) * 2)
    col_left = int((70+180) * 2)
    for i in range(0, shape[0]):
        for j in range(0, shape[1]):
            val = np.sum(bio1km[i*50:(i+1)*50, j*50:(j+1)*50])
            row_num = row_up + i
            col_num = col_left + j
            world_yield_zero[row_num,col_num] = val
    return world_yield_zero

### upscale_5arcTo30arc(arr_5arc)

In [14]:
def upscale_5arcTo30arc(arr_5arc):
    arr_30arc = np.zeros((360, 720))
    for i in range(0, 360):
        for j in range(0, 720):
            val = np.sum(arr_5arc[i*6: (i+1)*6, j*6: (j+1)*6])
            arr_30arc[i, j] = val
    return arr_30arc

# distance matrix

## cal_dis(lon_1, lat_1, lon_2, lat_2, resolution)

In [15]:
def cal_dis(lon_1, lat_1, lon_2, lat_2, resolution):
    dis = math.sqrt(pow((lon_1-lon_2),2)+pow((lat_1-lat_2),2)) * 111.1 / (60/resolution) # for 30 arcmin
    return dis    

## cal_dismatrix(df_c, df_cn, radiulimits_km, resolution)

In [16]:
def cal_dismatrix(df_c, df_cn, radiulimits_km, resolution):
    mxdis = pd.DataFrame(columns = ['c', 'cn', 'dis'])
    count = 0.
    for i in df_c.index: # len(grid_c_mapping)
        row_1 = df_c.loc[i, 'row']
        col_1 = df_c.loc[i, 'col']
        c_id_1 = int(df_c.loc[i, 'c_id'])
        for j in df_cn.index:  # len(grid_c_mapping)
            row_2 = df_cn.loc[j, 'row']
            col_2 = df_cn.loc[j, 'col']
            c_id_2 = int(df_cn.loc[j, 'c_id'])
            if (abs(row_2-row_1) <= radiulimits_km / 50) & (abs(col_2 - col_1) <= radiulimits_km / 50):  # filter 300 km
                dis = cal_dis(col_1, row_1, col_2, row_2, resolution)
                mxdis.loc[count] = [c_id_1, c_id_2, dis]
                count = count + 1
    # mxdis['c'] = mxdis['c'].astype(int)
    # mxdis['cn'] = mxdis['cn'].astype(int)
    return mxdis

# biomassproduction.gms

## landavailablebioen_bopf.gdx
columns: gridcell (1*3669), landuse (cropland, pasture, forest, othernatualland, builtup), decade, value <br>
description: the proportion of land use type in grid cell c in year t

In [17]:
filename = 'landavailablebioen_bopf'
nc_landcover = nc.Dataset("../input/raw_data/GLANDCOVER_30MIN.NC")

In [18]:
map_landcover = nc_landcover.variables['GLANDCOVER_30MIN'][:].data
name_arr = nc_landcover.variables['NGLNDCOV'][:].data
df_name = get_names(name_arr)
df_name.index

RangeIndex(start=0, stop=5, step=1)

In [ ]:
df_cid_landcover = pd.DataFrame()
for id_landcover in df_name.index:
    for year in range(2020, 2060, 10):
        type_land = df_name.loc[id_landcover, 'name']
        year_id = get_time_index(nc_landcover, year, 'time')
        map_arr = map_landcover[year_id, :, :, id_landcover]
        df_tmp = get_mapval_bycid(map_arr, c_id_mapping, type_land)
        df_tmp.loc[:, 'decade'] = [year for count in range(0, len(df_tmp))]
        df_cid_landcover = pd.concat([df_cid_landcover, df_tmp])

In [22]:
# df_cid_landcover.to_excel('../input/excel/%s.xlsx' % filename)
df_cid_landcover.head()

,Unnamed: 0,c_id,row,col,prov_code,prov_name,val,mapname,decade
0,0,0,74,603,118,Neimenggu,0.0,cropland,2020
1,1,1,74,604,111,Heilongjiang,0.0,cropland,2020
2,2,2,74,605,111,Heilongjiang,0.026477,cropland,2020
3,3,3,74,606,111,Heilongjiang,0.02737,cropland,2020
4,4,4,74,607,111,Heilongjiang,0.0,cropland,2020


In [23]:
# write gdx
df_para = df_cid_landcover.loc[:, ['c_id', 'mapname', 'decade', 'val']]
df_para.columns = ['c', 'l', 't', 'value']

In [24]:
dict_set = {
    'c' : list(pd.unique(df_para.loc[:, 'c'])),
    'l' : list(pd.unique(df_para.loc[:, 'l'])),
    't' : list(pd.unique(df_para.loc[:, 't']))
}
write_gdx(filename, dict_set, df_para, 'the proportion of land use type in grid cell c in year t')

## bpryields.gdx
columns: crop (ecrops, ewood), gridcell, decade, value <br>
description: the yield for each type of biomass resources in each grid cell c in year t

### yield of energy crops

In [19]:
filename = 'bpryields'
nc_yield = nc.Dataset('../input/raw_data/GRPPC_30MIN.NC')

In [20]:
# get crop name
namefile_arr = nc_yield.variables['NGFBFC'][:].data
df_name = get_names(namefile_arr)
df_name

,name
0,grass
1,RFWheat
2,RFRice
3,RFMaize
4,RFTropicalcereals
5,RFOthertemperatecereals
6,RFPulses
7,RFSoybeans
8,RFTemperateoilcrops
9,RFTropicaloilcrops


In [21]:
map_yield = nc_yield.variables['GRPPC_30MIN']

df_cid_yield_ecrop = pd.DataFrame()
for name_croptype in ['Woodybiofuel', 'Non-woodybiofuel']:
    for year in range(2020, 2070, 10):
        crop_index = list(df_name['name']).index(name_croptype)
        year_id = get_time_index(nc_yield, year, 'time')
        map_arr = map_yield[year_id, :, :, crop_index]
        df_tmp = get_mapval_bycid(map_arr, c_id_mapping, name_croptype)
        df_tmp.loc[:, 'decade'] = [year for count in range(0, len(df_tmp))]
        df_tmp.loc[:, 'val_GJ'] = df_tmp.loc[:, 'val'] * 19 # GJ/t
        df_cid_yield_ecrop = pd.concat([df_cid_yield_ecrop, df_tmp])

In [22]:
# df_cid_yield.to_excel('../input/excel/%s.xlsx' % filename)
df_cid_yield_ecrop.head()

,Unnamed: 0,c_id,row,col,prov_code,prov_name,val,mapname,decade,val_GJ
0,0,0,74,603,118,Neimenggu,756.391479,Woodybiofuel,2020,14371.43811
1,1,1,74,604,111,Heilongjiang,741.20105,Woodybiofuel,2020,14082.819946
2,2,2,74,605,111,Heilongjiang,739.016052,Woodybiofuel,2020,14041.304993
3,3,3,74,606,111,Heilongjiang,0.0,Woodybiofuel,2020,0.0
4,4,4,74,607,111,Heilongjiang,745.235474,Woodybiofuel,2020,14159.473999


In [23]:
df_val_ecrop = df_cid_yield_ecrop.loc[:, ['mapname', 'c_id', 'decade', 'val_GJ']]
df_val_ecrop.columns = ['r', 'c', 't', 'value']
df_val_ecrop = df_val_ecrop.replace('Woodybiofuel', 'ewood')
df_val_ecrop = df_val_ecrop.replace('Non-woodybiofuel', 'egrass')
df_val_ecrop

,r,c,t,value
0,ewood,0,2020,14371.43811
1,ewood,1,2020,14082.819946
2,ewood,2,2020,14041.304993
3,ewood,3,2020,0.0
4,ewood,4,2020,14159.473999
...,...,...,...,...
3664,egrass,3664,2060,73660.787354
3665,egrass,3665,2060,74941.060791
3666,egrass,3666,2060,73750.309082
3667,egrass,3667,2060,74776.990967


### yield of agricultural residues
$ Y_{agri} = \sum \limits _{r=wheat} ^{n} (y_{r} * rpr + \beta) * R_{r}$ <br>

Where, $R_{r}$ represent for the grid cell c, the share of this crop in the total amount of agricultural; $y_{r}$ represent the yield of agrigultural yield; $ Y_{agri} $ representes the final agricultural yields in the grid cell


In [24]:
share_agri = pd.read_excel('../input/raw_data/share_agri.xlsx', index_col = 0)
para_agrires = pd.read_excel('../input/raw_data/para_agrires.xlsx', sheet_name = 'Sheet1', index_col=0)

In [37]:
df_cid_yield_agrires = pd.DataFrame()
for year in range(2020, 2070, 10):
    rice_index = list(df_name['name']).index('RFRice')
    wheat_index = list(df_name['name']).index('RFWheat')
    maize_index = list(df_name['name']).index('RFMaize')
    year_id = get_time_index(nc_yield, year, 'time')
    
    # get the map of agricultural product
    map_arr_rice = map_yield[year_id, :, :, rice_index].data
    map_arr_wheat = map_yield[year_id, :, :, wheat_index].data
    map_arr_maize = map_yield[year_id, :, :, maize_index].data
    
    df_tmp_rice = get_mapval_bycid(map_arr_rice, c_id_mapping, 'riceRes_yield')
    df_tmp_wheat = get_mapval_bycid(map_arr_wheat, c_id_mapping, 'wheatRes_yield')
    df_tmp_maize = get_mapval_bycid(map_arr_maize, c_id_mapping, 'maizeRes_yield')
    
    rpr_rice = (np.log(df_tmp_rice.loc[:,'val'].replace(0, np.nan)) * para_agrires.loc['rice', 'rpr_a'] + para_agrires.loc['rice', 'rpr_b'])
    rpr_wheat = (np.log(df_tmp_wheat.loc[:,'val'].replace(0, np.nan)) * para_agrires.loc['temperate cereals', 'rpr_a'] + para_agrires.loc['temperate cereals', 'rpr_b'])
    rpr_maize = (np.log(df_tmp_maize.loc[:,'val'].replace(0, np.nan)) * para_agrires.loc['maize', 'rpr_a'] + para_agrires.loc['maize', 'rpr_b'])
    
    df_tmp_rice.loc[:, 'rice_GJ'] = rpr_rice * df_tmp_rice.loc[:,'val'] * para_agrires.loc['rice', 'moisture_content_percent']/100 * para_agrires.loc['rice', 'HHV_Gjperton'] * share_agri.loc['rice', 2020] 
    df_tmp_wheat.loc[:, 'wheat_GJ'] = rpr_wheat * df_tmp_wheat.loc[:,'val'] * para_agrires.loc['temperate cereals', 'moisture_content_percent']/100 * para_agrires.loc['temperate cereals', 'HHV_Gjperton'] * share_agri.loc['wheat', 2020] 
    df_tmp_maize.loc[:, 'maize_GJ'] = rpr_maize * df_tmp_maize.loc[:,'val'] * para_agrires.loc['maize', 'moisture_content_percent']/100 * para_agrires.loc['maize', 'HHV_Gjperton'] * share_agri.loc['maize', 2020] 

    # calculate the total agricultural residues
    df_agrires = df_tmp_rice.loc[:, ['c_id']]
    df_agrires.loc[:, 'val_GJ'] = np.array(df_tmp_rice.loc[:, 'rice_GJ'] + df_tmp_wheat.loc[:, 'wheat_GJ'] + df_tmp_maize.loc[:, 'maize_GJ'])
    df_agrires.loc[:, 'decade'] = [year for x in range(0, len(df_agrires))]
    df_agrires.loc[:, 'r'] = ['agrires' for x in range(0, len(df_agrires))]

    # merge
    df_cid_yield_agrires = pd.concat([df_cid_yield_agrires, df_agrires], axis=0)

In [38]:
df_cid_yield_agrires.columns = ['c', 'value', 't', 'r']
df_val_agri = df_cid_yield_agrires.loc[:, ['r', 'c', 't', 'value']]
df_val_agri.loc[df_val_agri.value.isna(), 'value'] = 0
df_val_agri

,r,c,t,value
0,agrires,0,2020,0
1,agrires,1,2020,0
2,agrires,2,2020,0
3,agrires,3,2020,0
4,agrires,4,2020,0
...,...,...,...,...
3664,agrires,3664,2060,1242.050958
3665,agrires,3665,2060,1118.540246
3666,agrires,3666,2060,1061.939042
3667,agrires,3667,2060,1105.846142


### yield of forestry residues 

In [39]:
name_croptype = 'Woodybiofuel'
df_cid_yield_foresres = pd.DataFrame()
for year in range(2020, 2070, 10):
    crop_index = list(df_name['name']).index(name_croptype)
    year_id = get_time_index(nc_yield, year, 'time')
    map_arr = map_yield[year_id, :, :, crop_index]
    df_tmp = get_mapval_bycid(map_arr, c_id_mapping, 'foresres')
    df_tmp.loc[:, 'decade'] = [year for count in range(0, len(df_tmp))]
    # forest residues parameters
    ratio = 0.3
    df_tmp.loc[:, 'val_GJ'] = df_tmp.loc[:, 'val'] * 19 * ratio # GJ/t
    df_cid_yield_foresres = pd.concat([df_cid_yield_foresres, df_tmp])

In [40]:
df_val_fores = df_cid_yield_foresres.loc[:, ['mapname', 'c_id', 'decade', 'val_GJ']]
df_val_fores.columns = ['r', 'c', 't', 'value']
df_val_fores

,r,c,t,value
0,foresres,0,2020,4311.431433
1,foresres,1,2020,4224.845984
2,foresres,2,2020,4212.391498
3,foresres,3,2020,0.0
4,foresres,4,2020,4247.8422
...,...,...,...,...
3664,foresres,3664,2060,8721.087671
3665,foresres,3665,2060,8447.024268
3666,foresres,3666,2060,8046.146997
3667,foresres,3667,2060,8352.213757


### generate yield dataset

In [41]:
df_val = pd.concat([df_val_agri, df_val_fores, df_val_ecrop], axis=0)
df_val.head()

,r,c,t,value
0,agrires,0,2020,0
1,agrires,1,2020,0
2,agrires,2,2020,0
3,agrires,3,2020,0
4,agrires,4,2020,0


In [42]:
# write gdx
dict_set = {
    'r' : list(pd.unique(df_val.loc[:, 'r'])),
    'c' : list(pd.unique(df_val.loc[:, 'c'])),
    't' : list(pd.unique(df_val.loc[:, 't']))
}

write_gdx(filename, dict_set, df_val, 'the yield of crop r in grid cell c in year t')

In [43]:
df_val.to_excel('../input/excel/bpryields.xlsx')

## bprcosts.gdx
(farmgatecost (US$/km2) + landrent) / yield + collection cost <br>
farmgate = 659 # USD/ha <br>
landrent = 20000 # unit? <br>
colleccost = 0.73 # US/GJ <br>
yield_val = 800 # t/km2 <br>
((farmgate/0.01) + landrent) / yield_val /16 + 0.73 <br>

In [44]:
filename = 'bprcosts'
nc_landrent = nc.Dataset('../input/raw_data/glandrent_30m.nc')

In [45]:
df_cid_landrent = pd.DataFrame()
for year in range(2020, 2070, 10):
    ls_year = list(nc_landrent.variables['year'][:].data)
    id_year = ls_year.index(year)
    map_landrent = nc_landrent.variables['glandrent_30m'][id_year, :, :].data.T
    df_tmp = get_mapval_bycid(map_landrent, c_id_mapping, 'landrent')
    df_tmp.loc[:, 'decade'] = [year for x in range(0, len(df_tmp))]
    df_cid_landrent = pd.concat([df_cid_landrent, df_tmp], axis=0)

In [46]:
# df_cid_landrent.to_excel('../input/excel/landrent.xlsx')

In [47]:
# process land rent
landrent_val = df_cid_landrent.loc[:, ['c_id', 'decade', 'val']]
landrent_val.columns = ['c', 't', 'landrent']
landrent_val_merge = landrent_val.set_index(['c', 't'])
farmgate = 659 # USD/ha
farmgate_res = 300 # USD/ha
collectcost = 0.73 # US/GJ
df_val = pd.read_excel('../input/excel/bpryields.xlsx')

In [48]:
# calculate the supply costs of egrass
egrass_val = df_val[df_val.r == 'egrass']
egrass_val_merge = egrass_val.set_index(['c','t'])
df_merge_egrass = pd.concat([egrass_val_merge, landrent_val_merge], axis=1)
df_merge_egrass.loc[:, 'sc_$pergj'] =  (df_merge_egrass.loc[:, 'landrent'] +  farmgate/0.01) / df_merge_egrass.loc[:, 'value'] + collectcost
df_merge_egrass = df_merge_egrass.reset_index()
out_egrass = df_merge_egrass.loc[:, ['r', 'c', 't', 'sc_$pergj']]

# calculate the supply costs of ewood
ewood_val = df_val[df_val.r == 'ewood']
ewood_val_merge = ewood_val.set_index(['c', 't'])
df_merge_ewood = pd.concat([ewood_val_merge, landrent_val_merge], axis=1)
df_merge_ewood.loc[:, 'sc_$pergj'] = (df_merge_ewood.loc[:, 'landrent'] +  farmgate/0.01) / df_merge_ewood.loc[:, 'value'] + collectcost
df_merge_ewood = df_merge_ewood.reset_index()
out_ewood = df_merge_ewood.loc[:, ['r', 'c', 't', 'sc_$pergj']]
out_ewood

,r,c,t,sc_$pergj
0,ewood,0,2020,5.315484
1,ewood,1,2020,5.606109
2,ewood,2,2020,5.99239
3,ewood,3,2020,inf
4,ewood,4,2020,5.750538
...,...,...,...,...
18340,ewood,3664,2060,5.554517
18341,ewood,3665,2060,6.381336
18342,ewood,3666,2060,6.70627
18343,ewood,3667,2060,5.24029


In [55]:
# calculate the supply costs of agricultural residues
agrires_val = df_val[df_val.r == 'agrires']
agrires_val.loc[:, 'sc_$pergj'] =  farmgate_res/0.01 / agrires_val.loc[:, 'value'] + collectcost
# df_merge_egrass
out_agrires = agrires_val.loc[:, ['r', 'c', 't', 'sc_$pergj']]

# calculate the supply costs of forestry residues
foresres_val = df_val[df_val.r == 'foresres']
foresres_val.loc[:, 'sc_$pergj'] = farmgate_res /0.01 / foresres_val.loc[:, 'value'] + collectcost
out_foresres = foresres_val.loc[:, ['r', 'c', 't', 'sc_$pergj']]
out_agrires

C:\Users\vicke\AppData\Local\Temp\ipykernel_44924\3605579445.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  agrires_val.loc[:, 'sc_$pergj'] =  farmgate_res/0.01 / agrires_val.loc[:, 'value'] + collectcost
C:\Users\vicke\AppData\Local\Temp\ipykernel_44924\3605579445.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  foresres_val.loc[:, 'sc_$pergj'] = farmgate_res /0.01 / foresres_val.loc[:, 'value'] + collectcost


,r,c,t,sc_$pergj
0,agrires,0,2020,inf
1,agrires,1,2020,inf
2,agrires,2,2020,inf
3,agrires,3,2020,inf
4,agrires,4,2020,inf
...,...,...,...,...
18340,agrires,3664,2060,24.883598
18341,agrires,3665,2060,27.550671
18342,agrires,3666,2060,28.980209
18343,agrires,3667,2060,27.858548


In [56]:
bprcosts = pd.concat([out_egrass, out_ewood, out_agrires, out_foresres], axis=0)
bprcosts.columns = ['r', 'c', 't', 'value']
bprcosts.loc[bprcosts.value>=9999, 'value'] = 9999
bprcosts

,r,c,t,value
0,egrass,0,2020,4.68742
1,egrass,1,2020,5.695668
2,egrass,2,2020,6.320025
3,egrass,3,2020,7.482873
4,egrass,4,2020,6.688565
...,...,...,...,...
36685,foresres,3664,2060,4.169938
36686,foresres,3665,2060,4.281547
36687,foresres,3666,2060,4.458493
36688,foresres,3667,2060,4.321862


In [57]:
bprcosts.to_excel('../input/excel/bprcosts.xlsx')

In [58]:
# write gdx
dict_set = {
    'r' : list(pd.unique(bprcosts.loc[:, 'r'])),
    'c' : list(pd.unique(bprcosts.loc[:, 'c'])),
    't' : list(pd.unique(bprcosts.loc[:, 't']))
}

write_gdx('bprcosts', dict_set, bprcosts, 'the unit production costs_dollarPerGj of resource r in grid cell c in time t')

In [1]:
# calculate the supply costs of residues

### plot supply costs curve

In [ ]:
# plot marginal supply costs for ecrops
# df_merge_egrass.sort_values(by=['sc_$pergj'], axis=0, ascending=True)

# df_merge_egrass_bprcosts = df_merge_egrass.reset_index()
# df_merge_egrass_bprcosts[df_merge_egrass_bprcosts]

## gcarea.gdx

In [187]:
nc_landarea = nc.Dataset('../input/raw_data/GAREA.NC')
nc_landarea.variables

{'longitude': <class 'netCDF4._netCDF4.Variable'>
 float64 longitude(longitude)
     standard_name: longitude
     units: degrees_east
 unlimited dimensions: 
 current shape = (4320,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'latitude': <class 'netCDF4._netCDF4.Variable'>
 float64 latitude(latitude)
     standard_name: latitude
     units: degrees_north
 unlimited dimensions: 
 current shape = (2160,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'time': <class 'netCDF4._netCDF4.Variable'>
 float32 time(time)
     standard_name: time
     units: days since 1970-01-01 00:00:00.0
     calender: gregorian
 unlimited dimensions: time
 current shape = (27,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'GAREA': <class 'netCDF4._netCDF4.Variable'>
 float32 GAREA(time, latitude, longitude)
     units: km2
     long_name: Grid area cell without area water, without builtup area
     qualifier: 
     _FillValue: -9999.0
 unlimited dimens

In [178]:
map_area = nc_landarea.variables['GAREA'][:]

In [182]:
df_area = pd.DataFrame()
for year in range(2020, 2070, 10):
    # first fetch time index;
    year_id = get_time_index(nc_landarea, year, 'time')
    map_arr_5arc = map_area[year_id, :, :].data
    map_arr_5arc[map_arr_5arc == -9999] = 0
    map_arr_30arc = upscale_5arcTo30arc(map_arr_5arc)
    df_tmp = get_mapval_bycid(map_arr_30arc, c_id_mapping, 'area')
    df_tmp.loc[:, 't'] = [year for x in range(0, len(df_tmp))]
    df_area = pd.concat([df_area, df_tmp], axis=0)
    # get the maps;
    # upscale the area;
    # get value by cid

In [184]:
# write as excel file
df_area.to_excel('../input/excel/gcarea.xlsx')

In [186]:
df_val = df_area.loc[:, ['c_id', 't', 'val']]
df_val.columns = ['c', 't', 'value']

In [188]:
# write gdx
dict_set = {
    'c' : list(pd.unique(df_val.loc[:, 'c'])),
    't' : list(pd.unique(df_val.loc[:, 't']))
}

write_gdx('gcarea', dict_set, df_val, 'the area (km2) for each grid cell c without water and built up area in different time t. ')

# carboncaptureandstorage.gms

## ccscap.gdx

In [212]:
ccscap = pd.read_excel('../input/raw_data/ccs.xlsx')
df_ccs =get_cid_byrowcol(ccscap, 'row', 'col', c_id_mapping)
df_val = df_ccs.loc[:, ['c_id', 'capacity_mt']]
df_val.columns = ['c', 'value']
df_val

,c,value
0,3139,9621.426621
1,789,22253.499587
2,2530,7500.001766
3,470,79608.185974
4,1682,12603.888046
5,2545,34543.616324
6,712,55307.888024
7,1353,144613.639892
8,1782,40234.531036
9,2029,55505.923733


In [202]:
# df_ccs_out.to_excel('../input/excel/ccscap.xlsx')
dict_set = {
    'c' : list(pd.unique(df_val.loc[:, 'c']))
}

write_gdx('ccscap', dict_set, df_val, 'ccs capacity in grid cell c')

# emission.gms

## efertilizers.gdx
this parameters can come from Li et al., EST, 2019

In [213]:
# make some assumption first
ef_agri = 0.
ef_foresres = 0.
ef_egrass = 550 # kgN2O/km2
ef_ewood = 10 # kgN2O/km2

# logistics.gms

## mxdis.gdx

In [26]:
c_id_mapping

,Unnamed: 0,c_id,row,col,prov_code,prov_name
0,0,0,74,603,118,Neimenggu
1,1,1,74,604,111,Heilongjiang
2,2,2,74,605,111,Heilongjiang
3,3,3,74,606,111,Heilongjiang
4,4,4,74,607,111,Heilongjiang
...,...,...,...,...,...,...
3664,3664,3664,142,579,108,Hainan
3665,3665,3665,142,580,108,Hainan
3666,3666,3666,143,578,108,Hainan
3667,3667,3667,143,579,108,Hainan


In [ ]:
df_val = cal_dismatrix(c_id_mapping.loc[:, :], c_id_mapping.loc[:, :], 1000000, 50)
# df_val.head()

In [17]:
# df_val.to_excel('../input/excel/mxdis.xlsx')`
df_val = pd.read_excel('../input/excel/mxdis_300km.xlsx')

In [24]:
df_val['c'] = df_val['c'].astype(int)
df_val['cn'] = df_val['cn'].astype(int)
df_val = df_val.drop(labels = 'Unnamed: 0', axis=1)

In [25]:
dict_set = {
    'c' : list(pd.unique(df_val.loc[:, 'c'])),
    'cn': list(pd.unique(df_val.loc[:, 'cn']))
}

write_gdx('mxdis', dict_set, df_val, 'distance matrix between grid cell c and cn')

# tortuosity

In [35]:
df_tortuosity = c_id_mapping.copy()
df_tortuosity.loc[:, 'value'] = 1.5
df_val= df_tortuosity.loc[:, ['c_id', 'value']]
df_val.columns = ['c', 'value']
df_val.to_excel('../input/excel/tortuosity.xlsx')

In [36]:
dict_set = {
    'c' : list(pd.unique(df_val.loc[:, 'c']))
}

write_gdx('tortuosity', dict_set, df_val, 'distance matrix between grid cell c and cn')

# target

## subsets

In [218]:
national_demand = pd.read_excel('../input/raw_data/nationalDemand.xlsx', index_col=0)
airport = pd.read_excel('../input/raw_data/demand_airport.xlsx', index_col=0)
harbor = pd.read_excel('../input/raw_data/demand_harbor.xlsx', index_col=0)
national_demand.head()

,2020,2030,2040,2050,2060,Unit
product,,,,,,
biojet,1.0,2.0,3.0,4.0,5.0,EJ
biomethanol,1.0,2.0,3.0,4.0,5.0,EJ
biojet_import,0.2,0.3,0.4,0.5,0.6,EJ
biomethanol_import,0.3,0.4,0.5,0.6,0.7,EJ
biojet_export,0.1,0.2,0.3,0.4,0.5,EJ


In [64]:
# produce air proxy
air_cid  = get_cid_byrowcol(airport, 'row', 'col', c_id_mapping)
air_sumbycid = air_cid.groupby('c_id').sum()
air_passenger = air_sumbycid.reset_index()
air_passenger= air_passenger[air_passenger.c_id!=""]
air_proxy_tmp = air_passenger.loc[:, ['c_id', 'PassegerThout_2019_per']]
air_proxy_tmp.loc[:, 'proxy'] = air_proxy_tmp.loc[:, 'PassegerThout_2019_per'] / np.sum(air_demand.loc[:, 'PassegerThout_2019_per'])
air_proxy = air_proxy_tmp.loc[:, ['c_id', 'proxy']]

604 103
605 121


In [74]:
# write airport proxy as subset gdx
dict_set = {
    'c' : list(pd.unique(air_proxy.loc[:, 'c_id']))
}

write_gdx('airport_proxy', dict_set, air_proxy, 'airport proxy')

In [72]:
# produce air proxy
harbor_cid  = get_cid_byrowcol(harbor, 'row', 'col', c_id_mapping)
harbor_sumbycid = harbor_cid.groupby('c_id').sum()
harbor_cargo = harbor_sumbycid.reset_index()
harbor_cargo= harbor_cargo[harbor_cargo.c_id!=""]
harbor_proxy_tmp = harbor_cargo.loc[:, ['c_id', 'CargoThrout_10_8t']]
harbor_proxy_tmp.loc[:, 'proxy'] = harbor_proxy_tmp.loc[:, 'CargoThrout_10_8t'] / np.sum(harbor_proxy_tmp.loc[:, 'CargoThrout_10_8t'])
harbor_proxy = harbor_proxy_tmp.loc[:, ['c_id', 'proxy']]
# harbor_proxy

604 103
605 121
579 141
587 137


In [75]:
# write harbor proxy as subset gdx
dict_set = {
    'c' : list(pd.unique(harbor_proxy.loc[:, 'c_id']))
}

write_gdx('harbor_proxy', dict_set, harbor_proxy, 'harbor proxy')

## tarbp.gdx

In [223]:
def cal_grid_target(df_proxy, national_demand, name_bioprod):
    df_val = pd.DataFrame()
    for year in range(2020, 2070, 10):
        df_tmp = df_proxy.copy()
        df_tmp.loc[:, 'target_gj'] = national_demand.loc[name_bioprod, year] * df_tmp.loc[:, 'proxy'] * pow(10,9) # E, P, T, G
        df_tmp.loc[:, 'decade'] = year
        df_val = pd.concat([df_val, df_tmp], axis=0)
    return df_val

In [225]:
df_grid_target = cal_grid_target(air_proxy, national_demand, 'biojet')
df_val = df_grid_target.loc[:, ['c_id', 'decade', 'target_gj']]
df_val.columns = ['c', 't', 'value']

## tarex.gdx

## tarim.gdx